<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display, HTML

# ==============================================================================
# 0. 데이터 불러오기 및 전처리 파편화 방지 설정
# ==============================================================================
df = pd.read_sas("/content/hn24_all.sas7bdat")

# 컬럼명을 문자열 변환 및 소문자 통일
df.columns = [
    col.decode("utf-8").lower() if isinstance(col, bytes) else col.lower()
    for col in df.columns
]
df = df.copy()

# ==============================================================================
# 1. 변수 재정의 및 전처리 (변수 정의표 기준 정확한 매핑)
# ==============================================================================
dm_cond = np.where(
    df["he_dm_hba1c"].isin([1, 2]), 0, np.where(df["he_dm_hba1c"] == 3, 1, np.nan)
)
hp_cond = np.where(
    df["he_hp"].isin([1, 2, 3]), 0, np.where(df["he_hp"] == 4, 1, np.nan)
)
sex_cond = np.where(df["sex"] == 2, 0, np.where(df["sex"] == 1, 1, np.nan))
age_cond = np.where(df["age"] >= 19, df["age"], np.nan)
obe_cond = np.where(
    df["he_obe"].isin([1, 2, 3]), 0, np.where(df["he_obe"].isin([4, 5, 6]), 1, np.nan)
)
incm_cond = np.where(df["incm"].isin([1, 2, 3, 4]), df["incm"], np.nan)
edu_cond = np.where(df["edu"].isin([1, 2, 3, 4]), df["edu"], np.nan)
smk_cond = np.where(
    df["bs3_1"].isin([3, 8]), 0, np.where(df["bs3_1"].isin([1, 2]), 1, np.nan)
)
drk_cond = np.where(
    df["bd1_11"].isin([1, 2, 8]),
    0,
    np.where(df["bd1_11"].isin([3, 4, 5, 6]), 1, np.nan),
)

new_cols = pd.DataFrame(
    {
        "dm": dm_cond,
        "hp": hp_cond,
        "sex_cat": sex_cond,
        "age_num": age_cond,
        "obe_cat": obe_cond,
        "incm_cat": incm_cond,
        "edu_cat": edu_cond,
        "smk_cat": smk_cond,
        "drk_cat": drk_cond,
    }
)
df = pd.concat([df, new_cols], axis=1)

# ==============================================================================
# 2. 도메인 분석용 표식 변수(Subpopulation Flag) 생성 (N = 5,567)
# ==============================================================================
df["domain"] = np.where(
    (df["age"] >= 19) & (df["dm"].notna()) & (df["hp"].notna()), 1, 0
)
sub_df = df[df["domain"] == 1].copy()


# ==============================================================================
# 3. 가중치 및 복합표본 설계를 반영한 통계량 산출 함수
# ==============================================================================
def get_weighted_pct(data, target_series, weight_series):
    valid_mask = target_series.notna() & weight_series.notna()
    if valid_mask.sum() == 0:
        return 0.0
    return (
        (target_series[valid_mask] * weight_series[valid_mask]).sum()
        / weight_series[valid_mask].sum()
    ) * 100


def make_row_stats(sub_df, var_name, cat_val, label):
    sub_df["target"] = (sub_df[var_name] == cat_val).astype(int)
    n_val = int(sub_df[sub_df["target"] == 1].shape[0])

    p_tot = get_weighted_pct(sub_df, sub_df["target"], sub_df["wt_itvex"])

    sub_dm1 = sub_df[sub_df["dm"] == 1]
    p_dm1 = get_weighted_pct(sub_dm1, sub_dm1["target"], sub_dm1["wt_itvex"])

    sub_dm0 = sub_df[sub_df["dm"] == 0]
    p_dm0 = get_weighted_pct(sub_dm0, sub_dm0["target"], sub_dm0["wt_itvex"])

    return {
        "특성": label,
        "n": n_val,
        "전체 (n=5,567)": f"{p_tot:.1f}",
        "당뇨병 유병군 (n=959)": f"{p_dm1:.1f}",
        "당뇨병 비유병군 (n=4,608)": f"{p_dm0:.1f}",
    }


def make_header_row(label):
    return {
        "특성": label,
        "n": "",
        "전체 (n=5,567)": "",
        "당뇨병 유병군 (n=959)": "",
        "당뇨병 비유병군 (n=4,608)": "",
    }


def make_miss_row(sub_df, var_name):
    n_miss = int(sub_df[sub_df[var_name].isna()].shape[0])
    return {
        "특성": "결측 수",
        "n": n_miss,
        "전체 (n=5,567)": "-",
        "당뇨병 유병군 (n=959)": "-",
        "당뇨병 비유병군 (n=4,608)": "-",
    }


def make_cont_row(sub_df, var_name, label):
    def get_mean_se(data):
        model = smf.wls(
            f"{var_name} ~ 1", data=data, weights=data["wt_itvex"]
        ).fit(cov_type="cluster", cov_kwds={"groups": data["psu"]})
        return f"{model.params['Intercept']:.1f} ({model.bse['Intercept']:.2f})"

    n_tot = int(sub_df[var_name].notna().sum())
    t_res = get_mean_se(sub_df)
    dm1_res = get_mean_se(sub_df[sub_df["dm"] == 1])
    dm0_res = get_mean_se(sub_df[sub_df["dm"] == 0])

    return {
        "특성": label,
        "n": n_tot,
        "전체 (n=5,567)": t_res,
        "당뇨병 유병군 (n=959)": dm1_res,
        "당뇨병 비유병군 (n=4,608)": dm0_res,
    }


# ==============================================================================
# 4. Table 1 최종 조립 및 HTML 테두리 표 출력
# ==============================================================================
table_rows = [
    make_header_row("고혈압 유병여부, n(%)"),
    make_row_stats(sub_df, "hp", 0, "비유병(정상/주의/전단계)"),
    make_row_stats(sub_df, "hp", 1, "유병(고혈압)"),
    make_cont_row(sub_df, "age_num", "연령(세), 가중평균(SE)"),
    make_header_row("성별, n(%)"),
    make_row_stats(sub_df, "sex_cat", 0, "여자"),
    make_row_stats(sub_df, "sex_cat", 1, "남자"),
    make_header_row("체질량지수 비만, n(%)"),
    make_row_stats(sub_df, "obe_cat", 0, "비만 아님 (1~3단계)"),
    make_row_stats(sub_df, "obe_cat", 1, "비만 (4~6단계)"),
    make_miss_row(sub_df, "obe_cat"),
    make_header_row("소득 4분위수, n(%)"),
    make_row_stats(sub_df, "incm_cat", 1, "하 (1)"),
    make_row_stats(sub_df, "incm_cat", 2, "중하 (2)"),
    make_row_stats(sub_df, "incm_cat", 3, "중상 (3)"),
    make_row_stats(sub_df, "incm_cat", 4, "상 (4)"),
    make_miss_row(sub_df, "incm_cat"),
    make_header_row("교육 수준, n(%)"),
    make_row_stats(sub_df, "edu_cat", 1, "초졸 이하 (1)"),
    make_row_stats(sub_df, "edu_cat", 2, "중졸 (2)"),
    make_row_stats(sub_df, "edu_cat", 3, "고졸 (3)"),
    make_row_stats(sub_df, "edu_cat", 4, "대졸 이상 (4)"),
    make_miss_row(sub_df, "edu_cat"),
    make_header_row("평생 흡연 여부, n(%)"),
    make_row_stats(sub_df, "smk_cat", 0, "비흡연 (과거/비흡연)"),
    make_row_stats(sub_df, "smk_cat", 1, "현재 흡연"),
    make_miss_row(sub_df, "smk_cat"),
    make_header_row("1년간 음주 빈도, n(%)"),
    make_row_stats(sub_df, "drk_cat", 0, "저위험 (월 1회 미만/안마심)"),
    make_row_stats(sub_df, "drk_cat", 1, "위험 (월 1회 이상)"),
    make_miss_row(sub_df, "drk_cat"),
]

table1_df = pd.DataFrame(table_rows)

# HTML 스타일을 적용하여 실선(테두리)이 있는 예쁜 표로 출력
html_table = table1_df.to_html(index=False, escape=False)
styled_html = f"""
<style>
    table {{ border-collapse: collapse; width: 100%; font-family: sans-serif; }}
    th, td {{ border: 1px solid #dddddd; text-align: left; padding: 8px; }}
    th {{ background-color: #f2f2f2; font-weight: bold; }}
    tr:nth-child(even) {{ background-color: #f9f9f9; }}
</style>
{html_table}
"""
display(HTML(styled_html))

특성,n,"전체 (n=5,567)",당뇨병 유병군 (n=959),"당뇨병 비유병군 (n=4,608)"
"고혈압 유병여부, n(%)",,,,
비유병(정상/주의/전단계),3577,69.3,35.6,75.1
유병(고혈압),1990,30.7,64.4,24.9
"연령(세), 가중평균(SE)",5567,49.7 (0.50),62.1 (0.63),47.6 (0.48)
"성별, n(%)",,,,
여자,3171,49.8,40.9,51.4
남자,2396,50.2,59.1,48.6
"체질량지수 비만, n(%)",,,,
비만 아님 (1~3단계),3489,61.4,44.9,64.2
비만 (4~6단계),2006,37.6,53.0,35.0


In [6]:
# ==============================================================================
# 5. 비가중 표본 통계량 추출 (B-3 과제용)
# ==============================================================================
def make_unweighted_row(sub_df, var_name, cat_val, label):
    target_mask = (sub_df[var_name] == cat_val)
    n_val = int(target_mask.sum())
    valid_mask = sub_df[var_name].notna()
    p_tot = (n_val / valid_mask.sum()) * 100 if valid_mask.sum() > 0 else 0
    return {"특성": label, "n": n_val, "비가중 전체 비율/평균": f"{p_tot:.1f}%"}

def make_unweighted_cont_row(sub_df, var_name, label):
    n_tot = int(sub_df[var_name].notna().sum())
    t_mean = sub_df[var_name].mean()
    return {"특성": label, "n": n_tot, "비가중 전체 비율/평균": f"{t_mean:.1f}"}

b3_rows = [
    {"특성": "--- 성별 ---", "n": "", "비가중 전체 비율/평균": ""},
    make_unweighted_row(sub_df, "sex_cat", 0, "여자"),
    make_unweighted_row(sub_df, "sex_cat", 1, "남자"),
    {"특성": "--- 교육 수준 ---", "n": "", "비가중 전체 비율/평균": ""},
    make_unweighted_row(sub_df, "edu_cat", 1, "초졸 이하 (1)"),
    make_unweighted_row(sub_df, "edu_cat", 4, "대졸 이상 (4)"),
    make_unweighted_cont_row(sub_df, "age_num", "연령(세) 평균"),
]
b3_df = pd.DataFrame(b3_rows)
display(b3_df)

,특성,n,비가중 전체 비율/평균
0,--- 성별 ---,,
1,여자,3171,57.0%
2,남자,2396,43.0%
3,--- 교육 수준 ---,,
4,초졸 이하 (1),882,16.2%
5,대졸 이상 (4),2209,40.6%
6,연령(세) 평균,5567,54.0
